# Swin v2 temporal full training

Runs fold0 for 8 epochs with temporal differences plus temporal mean/std image channels.
Checkpoints are saved under `/kaggle/working/models/swin_v2_temporal_stable`.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys


def install_p100_torch_if_needed():
    try:
        gpu = subprocess.check_output(
            ['bash', '-lc', "nvidia-smi --query-gpu=name --format=csv,noheader | head -1"],
            text=True,
        ).strip()
    except Exception:
        gpu = ''
    if 'P100' in gpu:
        print(f'Installing a Pascal-compatible PyTorch build for {gpu}')
        subprocess.run(
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '-q',
                '--force-reinstall',
                'torch==2.5.1',
                'torchvision==0.20.1',
                '--index-url',
                'https://download.pytorch.org/whl/cu121',
            ],
            check=True,
        )
        subprocess.run(
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '-q',
                '--force-reinstall',
                'pillow==11.3.0',
            ],
            check=True,
        )
    return gpu


gpu_name = install_p100_torch_if_needed()

REPO = Path('/kaggle/working/solafune-nowcast')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/shionsuio/solafune-nowcast.git', str(REPO)], check=True)

sys.path.insert(0, str(REPO / 'src'))

import torch
print('GPU:', gpu_name or torch.cuda.get_device_name(0), 'torch:', torch.__version__)
assert torch.cuda.is_available(), 'GPU is required for this notebook'

from run_swin_temporal_full import run

DATA_ROOT = Path('/kaggle/input/solafune-dataset-v2')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('/kaggle/input/datasets/suioshion/solafune-dataset-v2')

# Existing full Swin stats can be reused because temporal channels are derived
# after per-band normalization; raw band statistics are unchanged.
STATS_DATASET = Path('/kaggle/input/solafune-stat')
if not STATS_DATASET.exists():
    STATS_DATASET = Path('/kaggle/input/datasets/suioshion/solafune-stat')
BAND_STATS_ROOT = str(STATS_DATASET) if STATS_DATASET.exists() else None
print('DATA_ROOT:', DATA_ROOT)
print('BAND_STATS_ROOT:', BAND_STATS_ROOT)

class Args:
    root = '/kaggle/working'
    kaggle_input_root = str(DATA_ROOT)
    folds = '0'
    epochs = 8
    batch_size = 8
    workers = 2
    lr_encoder = 2e-5
    lr_head = 1e-4
    stats_samples_per_satellite = 1500
    seed = 42
    model_subdir = 'swin_v2_temporal_stable'
    band_stats_root = BAND_STATS_ROOT
    no_pretrained = False
    no_amp = True

summary_path = run(Args())

import pandas as pd
summary = pd.read_csv(summary_path)
display(summary)

history_path = Path('/kaggle/working/models/swin_v2_temporal_stable/history_fold0.csv')
history = pd.read_csv(history_path)
display(history)

ax = history.plot(x='epoch', y='validation_rmse', marker='o', figsize=(8, 4), title='Swin v2 temporal stable fold0 validation RMSE')
ax.set_ylabel('RMSE')
